# 02. Предобработка данных

## Импорт библиотек

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd

## Загрузка данных

In [2]:
# Добавляем корневую папку проекта в sys.path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data_loader import DataLoader

In [3]:
# Загружаем переменные из .env
load_dotenv()

# Загружаем датасет
DATASET_PATH = os.getenv("DATA_PATH")
KAGGLE_DS = os.getenv("KAGGLE_DATASET")

df = DataLoader.load(DATASET_PATH, KAGGLE_DS)

In [4]:
df.head()

,client_ID,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,...,city_latitude,city_longitude,employment_type,loan_term_months,loan_to_income_ratio,other_debt,debt_to_income_ratio,open_accounts,credit_utilization_ratio,past_delinquencies
0,CUST_00001,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,...,43.6532,-79.3832,Self-employed,36,0.593220,8402.453850,0.735635,14,0.495557,0
1,CUST_00002,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,...,43.6532,-79.3832,Full-time,36,0.104167,1607.802794,0.271646,10,0.585436,3
2,CUST_00003,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,...,51.6214,-3.9436,Full-time,36,0.572917,2760.505633,0.860469,14,0.750732,0
3,CUST_00004,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,...,49.2827,-123.1207,Part-time,12,0.534351,7155.286150,0.643592,15,0.379333,0
4,CUST_00005,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,...,42.8864,-78.8784,Part-time,36,0.643382,15626.153440,0.930628,4,0.228103,0


## Удаление идентификаторов и мультиколлинеарных признаков

In [5]:
# Удаление id
df.drop(columns=["client_ID"], inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  str    
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  str    
 5   loan_grade                  32581 non-null  str    
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  str    
 11  cb_person_cred_hist_length  32581 non-null  int64  
 12  gender                      32581 non-null  str    
 13  marital_status              32581 non-null

In [6]:
# Удаление одного из дублирующихся признаков
df.drop(columns=["loan_to_income_ratio"], inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 27 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  str    
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  str    
 5   loan_grade                  32581 non-null  str    
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  str    
 11  cb_person_cred_hist_length  32581 non-null  int64  
 12  gender                      32581 non-null  str    
 13  marital_status              32581 non-null

In [7]:
# Удаление избыточных географических признаков
df.drop(columns=["city_latitude", "city_longitude", "state", "country"], inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  str    
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  str    
 5   loan_grade                  32581 non-null  str    
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  str    
 11  cb_person_cred_hist_length  32581 non-null  int64  
 12  gender                      32581 non-null  str    
 13  marital_status              32581 non-null

## Обработка выбросов

In [8]:
# Статисктика по возрасту
df["person_age"].describe()

count    32581.000000
mean        27.734600
std          6.348078
min         20.000000
25%         23.000000
50%         26.000000
75%         30.000000
max        144.000000
Name: person_age, dtype: float64

In [9]:
# Топ - 10 самых больших возрастов
df["person_age"].nlargest(10)

81       144
183      144
32297    144
575      123
747      123
32416     94
32506     84
32422     80
32355     78
32534     76
Name: person_age, dtype: int64

In [10]:
# Принято решение удалить строки с возрастом > 100, 
# так как это явная ошибка
df.query("person_age <= 100", inplace=True)
df["person_age"].describe()

count    32576.000000
mean        27.718044
std          6.204990
min         20.000000
25%         23.000000
50%         26.000000
75%         30.000000
max         94.000000
Name: person_age, dtype: float64

In [11]:
# Статистика по длине истории занятости
df["person_emp_length"].describe()

count    31681.000000
mean         4.789527
std          4.142706
min          0.000000
25%          2.000000
50%          4.000000
75%          7.000000
max        123.000000
Name: person_emp_length, dtype: float64

In [12]:
# Топ - 10 самых больших историй занятости
df["person_emp_length"].nlargest(10)

0        123.0
210      123.0
32355     41.0
32515     38.0
32428     34.0
30914     31.0
31866     31.0
31867     31.0
32263     31.0
32539     30.0
Name: person_emp_length, dtype: float64

In [13]:
# Удаление строк с person_emp_length > 50,
# так как таких значений всего 2 и значения у обоих 123,
# это явная ошибка
df = df[(df["person_emp_length"] <= 50) | df["person_emp_length"].isna()]
df["person_emp_length"].describe()

count    31679.000000
mean         4.782064
std          4.034948
min          0.000000
25%          2.000000
50%          4.000000
75%          7.000000
max         41.000000
Name: person_emp_length, dtype: float64

In [14]:
# Статистика по доходу
df["person_income"].describe()

count    3.257400e+04
mean     6.587848e+04
std      5.253194e+04
min      4.000000e+03
25%      3.850000e+04
50%      5.500000e+04
75%      7.920000e+04
max      2.039784e+06
Name: person_income, dtype: float64

In [15]:
# Топ - 10 с самым высоким доходом
# Так как такой доход вполне может быть,
# принято решение оставить этот признак как есть
# Для линейной регрессии возможно придется применить логарифмическое преобразование
# Для деревьев оставить признак как есть
df["person_income"].nlargest(10)

30049    2039784
32546    1900000
32497    1782000
31924    1440000
31922    1362000
17833    1200000
29119    1200000
29120    1200000
17834     948000
29121     900000
Name: person_income, dtype: int64

In [16]:
# Статистика после удаления выбросов
df.describe()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length,loan_term_months,other_debt,debt_to_income_ratio,open_accounts,credit_utilization_ratio,past_delinquencies
count,32574.000000,3.257400e+04,31679.000000,32574.000000,29459.000000,32574.000000,32574.000000,32574.000000,32574.000000,32574.000000,32574.000000,32574.000000,32574.000000,32574.000000
mean,27.718426,6.587848e+04,4.782064,9588.018051,11.011529,0.218180,0.170202,5.804108,38.502118,11528.852140,0.345203,8.042119,0.499900,0.505188
std,6.204987,5.253194e+04,4.034948,6320.249598,3.240497,0.413017,0.106755,4.053873,16.013291,11315.351413,0.129391,4.328086,0.259528,0.711783
min,20.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000,0.000000,2.000000,12.000000,225.207376,0.064502,0.000000,0.050001,0.000000
25%,23.000000,3.850000e+04,2.000000,5000.000000,7.900000,0.000000,0.090000,3.000000,24.000000,5385.828062,0.251245,4.000000,0.275362,0.000000
50%,26.000000,5.500000e+04,4.000000,8000.000000,10.990000,0.000000,0.150000,4.000000,36.000000,8994.217919,0.333190,8.000000,0.500326,0.000000
75%,30.000000,7.920000e+04,7.000000,12200.000000,13.470000,0.000000,0.230000,8.000000,60.000000,14559.669650,0.423138,12.000000,0.725060,1.000000
max,94.000000,2.039784e+06,41.000000,35000.000000,23.220000,1.000000,0.830000,30.000000,60.000000,399125.753000,1.053888,15.000000,0.949998,6.000000


## Обработка пропусков

In [17]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    "Кол-во пропусков": missing,
    "Доля (%)": missing_pct
}).sort_values("Кол-во пропусков", ascending=False)
missing_df = missing_df[missing_df["Кол-во пропусков"] > 0]
print("Пропуски до обработки:")
display(missing_df)

Пропуски до обработки:


,Кол-во пропусков,Доля (%)
loan_int_rate,3115,9.562842
person_emp_length,895,2.747590


In [18]:
# Вычисляем медианы ставки по каждому классу loan_grade
rate_medians = df.groupby("loan_grade")["loan_int_rate"].median()
print("Медианы loan_int_rate по loan_grade:")
print(rate_medians)

# Заполняем пропуски
def fill_rate(row):
    if pd.isna(row["loan_int_rate"]):
        return rate_medians[row["loan_grade"]]
    return row["loan_int_rate"]

df["loan_int_rate"] = df.apply(fill_rate, axis=1)

Медианы loan_int_rate по loan_grade:
loan_grade
A     7.490
B    10.990
C    13.480
D    15.310
E    16.820
F    18.535
G    20.160
Name: loan_int_rate, dtype: float64


In [19]:
# Индикатор пропуска
df["emp_length_missing"] = df["person_emp_length"].isna().astype(int)

# Медианная импутация
median_emp = df["person_emp_length"].median()
df.fillna({'person_emp_length': median_emp}, inplace=True)
print(f"person_emp_length заполнена медианой = {median_emp:.2f}")
print(f"Доля пропусков (индикатор): {df['emp_length_missing'].mean():.2%}")

person_emp_length заполнена медианой = 4.00
Доля пропусков (индикатор): 2.75%


In [20]:
df.info()

<class 'pandas.DataFrame'>
Index: 32574 entries, 1 to 32580
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32574 non-null  int64  
 1   person_income               32574 non-null  int64  
 2   person_home_ownership       32574 non-null  str    
 3   person_emp_length           32574 non-null  float64
 4   loan_intent                 32574 non-null  str    
 5   loan_grade                  32574 non-null  str    
 6   loan_amnt                   32574 non-null  int64  
 7   loan_int_rate               32574 non-null  float64
 8   loan_status                 32574 non-null  int64  
 9   loan_percent_income         32574 non-null  float64
 10  cb_person_default_on_file   32574 non-null  str    
 11  cb_person_cred_hist_length  32574 non-null  int64  
 12  gender                      32574 non-null  str    
 13  marital_status              32574 non-null  str

## Проверка препроцессинга из preprocess.py

In [21]:
df = DataLoader.load(DATASET_PATH, KAGGLE_DS)

In [22]:
from src.preprocess import Preprocessor

prep = Preprocessor()
df_cleaned = prep.fit_transform(df)

print(f"Размер после обработки: {df_cleaned.shape}")
print(f"Пропуски: {df_cleaned.isnull().sum().sum()}")

Размер после обработки: (32581, 24)
Пропуски: 0


In [23]:
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  str    
 3   person_emp_length           32581 non-null  float64
 4   loan_intent                 32581 non-null  str    
 5   loan_grade                  32581 non-null  str    
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               32581 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  str    
 11  cb_person_cred_hist_length  32581 non-null  int64  
 12  gender                      32581 non-null  str    
 13  marital_status              32581 non-null

## Выводы по предобработке данных

В результате выполнения предобработки данных были выполнены следующие шаги:

---

### 1. Удаление идентификаторов и мультиколлинеарных признаков
- Удалён столбец `client_ID` (не несёт predictive-ценности).
- Удалён дублирующий признак `loan_to_income_ratio` (оставлен `loan_percent_income`).
- Удалены избыточные географические признаки: `country`, `state`, `city_latitude`, `city_longitude` (оставлен только `city`).

**Итог:** количество признаков уменьшилось с 29 до 23.

---

### 2. Обработка выбросов
- **Возраст (`person_age`)**: удалены строки с возрастом > 100 (явные ошибки). Исходный максимум 144 → после обработки 94.
- **Стаж работы (`person_emp_length`)**: удалены строки со стажем > 50 (всего 2 записи с 123 годами). Максимум стал 41.
- **Доход (`person_income`)**: оставлен без изменений, так как значения > 1 млн возможны и не являются ошибками.

**Итог:** удалено 7 строк (из 32581 → 32574). Выбросы в других признаках (`loan_amnt`, `other_debt` и др.) оставлены для дальнейшей обработки на этапе feature engineering.

---

### 3. Обработка пропусков
- **`loan_int_rate`**: заполнена медианными значениями по группам `loan_grade` (медианы: A=7.49, B=10.99, C=13.48, D=15.31, E=16.82, F=18.535, G=20.16). Обработано 3115 пропусков.
- **`person_emp_length`**: создан индикатор `emp_length_missing` (1, если было пропущено). Сами пропуски заполнены глобальной медианой (4.0). Обработано 895 пропусков.

**Итог:** все пропуски устранены.

---

### 4. Итоговая структура данных
- **Размер**: 32 574 строки, 24 колонки.
- **Память**: ~6.2 MB.
- **Типы данных**: 6 float64, 9 int64, 9 object (строковые категории).
- **Новые признаки**: добавлен `emp_length_missing`.
- **Пропуски**: отсутствуют.

Данные полностью готовы для следующего этапа — **создания новых признаков (Feature Engineering)** и **обучения моделей**.